# Import Dependencies

In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip show mesa
print('\nok')

Name: Mesa
Version: 3.5.2
Summary: Agent-based modeling (ABM) in Python
Home-page: https://github.com/mesa/mesa
Author: 
Author-email: Mesa Team <maintainers@projectmesa.dev>
License: Apache 2.0
Location: /home/eonz/.venvs/main/lib/python3.14/site-packages
Requires: numpy, pandas, scipy, tqdm
Required-by: 

ok


# Resource Classes

In [28]:
class Sugar(mesa.Agent):
    def __init__(self, model, max_sugar):
        # inherit mesa.agent uid and mode type
        super().__init__(model= model)
        # set other param
        self.amount = max_sugar
        self.max_sugar = max_sugar
    def step(self):
        self.amount = min([self.amount + 1, self.max_sugar])

In [29]:
class Spice(mesa.Agent):
    def __init__(self, model, max_spice):
        # inherit mesa.agent uid and mode type
        super().__init__(model= model)
        # set other param
        self.amount = max_spice
        self.max_spice = max_spice
    def step(self):
        self.amount = min([self.amount + 1, self.max_spice])

# Trader Agent

In [11]:
class Trader(mesa.Agent): 
    def __init__(
        self,
        model,
        moore,
        sugar,
        spice,
        metabolism_sugar, metabolism_spice,
        vision
    ):

        super().__init__(model = model)
        self.moore = False,
        self.sugar = sugar,
        self.spice = spice,
        self.metabolism_sugar = metabolism_sugar,
        self.metabolism_spice = metabolism_spice,
        self.vision = vision

    
        

# Model Class

In [45]:
class SugarscapeG1mt(mesa.Model):
    """
    Manager class to run Sugarscape with Traders
    """
    def __init__(
        self, w=50, h=50,
        initial_population = 200,
        endowment_min = 25, endowment_max = 50,
        metabolism_min = 1, metabolism_max = 5,
        vision_min = 1, vision_max = 5
        
    ):
        super().__init__()
        
        # initialize parameter attribute
        
        # range set to make the agent heterogenous
        # heterogenous is one of key aspect of agent based modeling
        # such that novel behavioour can produced
        self.width = w
        self.height = h
        self.initial_population = initial_population
        self.endowment_min = endowment_min
        self.endowment_max = endowment_max
        self.metabolism_min = metabolism_min
        self.metabolism_max = metabolism_max
        self.vision_min = vision_min
        self.vision_max = vision_max
        

        # initialize mesa grid
        self.grid = mesa.space.MultiGrid(
            width=self.width,
            height=self.height,
            torus=False
          )

        sugar_dist = np.genfromtxt('sugar-map.txt')
        spice_dist = np.flip(sugar_dist, 1)

        # assign resources agents to each grid
        for _, (x, y) in self.grid.coord_iter():
            max_sugar = sugar_dist[x, y]
            max_spice = spice_dist[x, y]

            if max_sugar > 0:
                # initialze the agent
                sugar = Sugar(self, max_sugar)
                # put the the agent to curr pos
                self.grid.place_agent(sugar, (x, y))

            if max_spice > 0:
                spice = Spice(self, max_spice)
                self.grid.place_agent(spice, (x, y))

        # trading agents  
        for i in range(self.initial_population):
            # get agent pos
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)

            # The Artificial societies

            # initial endowment 
            sugar  = int(self.random.uniform(self.endowment_min, self.endowment_max + 1))
            spice  = int(self.random.uniform(self.endowment_min, self.endowment_max + 1))

            # initial metabolism
            metabolism_sugar = int(self.random.uniform(self.metabolism_min, self.metabolism_max + 1))
            metabolism_spice = int(self.random.uniform(self.metabolism_min, self.metabolism_max + 1)) 

            # initial vision
            vision = int(self.random.uniform(self.vision_min, self.vision_max + 1))

            # initialize the trader aget
            trader = Trader(
                self,
                moore = False,
                sugar = sugar, spice = spice,
                metabolism_sugar = metabolism_sugar,
                metabolism_spice = metabolism_spice,
                vision = vision
            )

            # place trader agent
            self.grid.place_agent(trader, (x, y))
            
            # data collector for recording steps track
            self.datacollector = mesa.datacollection.DataCollector(
                model_reporters={
                    # Track the current step count or model-level metrics
                    "Step": lambda m: m.steps,
                    # Track the total number of agents
                    "AgentCount": lambda m: len(m.agents),
                },
                
                agent_reporters={
                    # Track individual agent attributes per step
                    "AgentID": "unique_id"
                },
            )
            
    # intialize schedluer + step function
    def step(self):
        self.datacollector.collect(self)
        self.agents.select(agent_type=Trader).shuffle_do("step")
        self.agents.select(agent_type=Sugar).shuffle_do("step")
        self.agents.select(agent_type=Spice).shuffle_do("step")


    def run(self, steps = 1000):
        for i in range(steps):
            self.step()

# Run Sugarscape

In [46]:
model = SugarscapeG1mt()
model.run()

model_dfs = model.datacollector.get_model_vars_dataframe()
print(model_dfs)

     Step  AgentCount
0       1        4338
1       2        4338
2       3        4338
3       4        4338
4       5        4338
..    ...         ...
995   996        4338
996   997        4338
997   998        4338
998   999        4338
999  1000        4338

[1000 rows x 2 columns]
